# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}\n")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"Version: {getattr(metadata, 'version', '')}\n")
print(f"License: {getattr(metadata, 'license', '')}")


## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their Croissant `@id` field.

Let's enumerate all the available `recordSet` objects in the metadata along with their fields and columns.

In [ ]:
# Helper function to pretty print information about each record set
def print_record_sets(metadata):
    # Some Croissant metadata objects have a .recordSet attribute; check its type
    record_sets = getattr(metadata, 'recordSet', None)
    if record_sets is None or len(record_sets) == 0:
        print('No record sets available in this dataset.')
        return []

    ids = []
    for rs in record_sets:
        rsid = getattr(rs, '@id', None)
        print(f'RECORD SET @id: {rsid}')
        print(f'  Name: {getattr(rs, "name", "--")}, Description: {getattr(rs, "description", "--")[:120]}')
        # Fields
        fields = getattr(rs, 'field', getattr(rs, 'fields', []))
        if not isinstance(fields, list):
            fields = [fields]
        print('  Fields:')
        for f in fields:
            fname = getattr(f, 'name', '') if hasattr(f, 'name') else ''
            fid = getattr(f, '@id', '') if hasattr(f, '@id') else ''
            fdesc = getattr(f, 'description', '')[:60] if hasattr(f, 'description') else ''
            print(f'    - @id: {fid}, name: {fname}, description: {fdesc}')
        # Columns
        columns = getattr(rs, 'column', getattr(rs, 'columns', []))
        if not isinstance(columns, list):
            columns = [columns]
        if columns:
            print('  Columns:')
            for c in columns:
                cname = getattr(c, 'name', '') if hasattr(c, 'name') else ''
                cid = getattr(c, '@id', '') if hasattr(c, '@id') else ''
                cdesc = getattr(c, 'description', '')[:60] if hasattr(c, 'description') else ''
                print(f'    - @id: {cid}, name: {cname}, description: {cdesc}')
        print()
        ids.append(rsid)
    return ids

record_set_ids = print_record_sets(metadata)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For demonstration, we'll extract data from every available record set, if any.

In [ ]:
# Extract data from each record set
if not record_set_ids:
    print('No record sets available to extract records.')
else:
    dataframes = {}
    for record_set_id in record_set_ids:
        print(f"Extracting records from record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Fields (columns) for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head(), '\n')


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note**: If no record sets are available, this section will be skipped. Otherwise, select one record set and field by `@id`.

In [ ]:
import numpy as np

if record_set_ids:
    current_record_set = record_set_ids[0]  # Just as an example pick the first one
    df = dataframes[current_record_set]
    print(f"Working with record set: {current_record_set}")
    
    # Find a numeric field in the record set (by checking dtype after loading)
    numeric_col = None
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_col = numeric_candidates[0]
        print(f"Numeric field selected for analysis: {numeric_col}")
    else:
        # Try to infer numerics (sometimes loaded as str, try conversion)
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                continue
        numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_candidates:
            numeric_col = numeric_candidates[0]
            print(f"Numeric field (coerced) selected for analysis: {numeric_col}")
            # Optionally drop rows that are all NaN for this col
            df = df.dropna(subset=[numeric_col])
        else:
            print('No numeric columns found for EDA.')
    
    if numeric_col:
        threshold = df[numeric_col].mean()
        filtered_df = df[df[numeric_col] > threshold]
        print(f"Filtered records with {numeric_col} > {threshold:.2f} (mean value):")
        print(filtered_df.head())
        
        norm_col = f"{numeric_col}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
        print(f"\nNormalized {numeric_col} for filtered records (z-score normalization):")
        print(filtered_df[[numeric_col, norm_col]].head())
        
        # Try to group by the first non-numeric column
        group_field = None
        for col in df.columns:
            if col != numeric_col and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_col].mean().reset_index()
            print(f"\nGrouped mean {numeric_col} by {group_field}:")
            print(grouped_df.head())
    
else:
    print("No record sets available; skipping EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

**This example plots a histogram of the selected numeric field in the record set, if available.**

In [ ]:
import matplotlib.pyplot as plt

if record_set_ids and numeric_col:
    plt.figure(figsize=(7, 4))
    df[numeric_col].hist(bins=30, color='skyblue', edgecolor='black')
    plt.title(f'Histogram of {numeric_col} in record set {current_record_set}')
    plt.xlabel(numeric_col)
    plt.ylabel('Frequency')
    plt.show()

    # If there's a group, visualize boxplot by group
    if group_field:
        plt.figure(figsize=(8, 4))
        df.boxplot(column=numeric_col, by=group_field, grid=False)
        plt.title(f'{numeric_col} by {group_field}')
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_col)
        plt.xticks(rotation=30)
        plt.show()
else:
    print('No numeric record field to visualize.')


## 6. Conclusion
In this notebook, we've loaded the dataset metadata, enumerated record sets and their fields (by `@id`), extracted data, and performed basic exploratory and visual analyses.

- All data access and operations reference entities by their Croissant `@id` field, ensuring consistency and reproducibility.
- You can further build upon this notebook by refining the EDA, adding new visualizations, and connecting downstream ML workflows.

**If no record sets are present, consult the dataset documentation or Croissant metadata for additional instructions or asset download references.**